# ZaenDrive — Week 7 Customer Service Agent
**C115451101 — Zaen Alaabden Abdul Rahim**

Run the cells in order. A CPU runtime is enough. The model runs through your configured API.
Use your own course API key; never paste it into chat or a visible code cell.

The project is implemented, but the actual model score and report screenshots must come from your own live run.


## 1. Upload the project ZIP
Select **C115451101_Zaen_CustomerService_Project.zip**, not the teacher's original assignment ZIP.
This cell checks archive paths before extracting and starts in the correct project folder.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import os, sys

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one project ZIP.")
filename, archive_bytes = next(iter(uploaded.items()))
if not filename.lower().endswith(".zip"):
    raise ValueError("Choose the project ZIP file.")
extract_root = Path("/content/zaendrive_w07")
extract_root.mkdir(exist_ok=True)
with ZipFile(BytesIO(archive_bytes)) as archive:
    for member in archive.infolist():
        target = (extract_root / member.filename).resolve()
        if not target.is_relative_to(extract_root.resolve()):
            raise ValueError("Unsafe ZIP path.")
    if not any(name.endswith("ZaenDrive/rental_tools.py") for name in archive.namelist()):
        raise ValueError("This is not the ZaenDrive project ZIP.")
    archive.extractall(extract_root)
PROJECT_DIR = extract_root / "ZaenDrive"
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Project ready:", PROJECT_DIR)
del uploaded, archive_bytes


## 2. Install packages
If Colab requests a session restart, restart it, rerun this cell, then continue with cell 3.
The path is set again here so the notebook also works after a session restart.


In [ ]:
from pathlib import Path
import os, sys, subprocess
PROJECT_DIR = Path("/content/zaendrive_w07/ZaenDrive")
if not (PROJECT_DIR / "app.py").exists():
    raise FileNotFoundError("Run the ZIP upload cell first.")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed. If Colab asks for a restart, restart and rerun this cell.")


## 3. Configure the API — secret input
The URL and model below are taken from the teacher's example. They have not yet been verified with your account.
If you use another tool-capable provider, change only its URL and model here.

Option A: add a Colab secret named **NKUST_API_KEY** in the key-shaped sidebar and allow this notebook to read it.
Option B: enter the key in the hidden prompt. It is kept in the current runtime's environment, not written into the notebook or project.


In [ ]:
import os
from getpass import getpass

os.environ["LLM_API_URL"] = "https://www.iai.nkust.edu.tw/aihub/v1/"
os.environ["MODEL_NAME"] = "Furen-large"
os.environ["MAX_STEPS"] = "6"
os.environ["MIN_CALL_INTERVAL"] = "5"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

api_key = ""
try:
    from google.colab import userdata
    api_key = userdata.get("NKUST_API_KEY") or ""
except Exception:
    pass
if not api_key.strip():
    api_key = getpass("Enter your course API key (hidden): ")
if not api_key.strip():
    raise ValueError("No API key was entered. Obtain the correct key from your course setup.")
os.environ["API_KEY"] = api_key.strip()
del api_key
print("API key loaded without displaying it. Model:", os.environ["MODEL_NAME"])


## 4. Local code checks — no API requests
These 14 tests check Python logic using scripted responses. **They are not the assignment's live tool-choice score.**


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "unittest", "-v", "test_offline"], check=True)


## 5. Start the chat
Open the displayed link. First ask **What are your opening hours?** and check that the tool panel shows `get_rental_info`.
Then try the four screenshot scenarios from `START_HERE_DE.md`.
The share link opens this demo while the Colab runtime is active.


In [ ]:
import gradio as gr
from rental_tools import initialize_database
from agent import Settings, RentalAgent, make_client
from app import build_app

gr.close_all()
print("Database rows:", initialize_database())
settings = Settings.from_env()
agent = RentalAgent(make_client(settings), settings)
demo = build_app(agent)
demo.queue().launch(share=True, prevent_thread_lock=True)


## 6. REAL tool-choice test — run after one chat question succeeds
This cell makes real API requests for 14 questions, with a pause between model calls. Do not use the chat at the same time.
Keep the first result. If you actually change the code/prompt, rerun with `RUN_LABEL = "after-fix"`.
An incomplete API-error run is not a final score. Take a screenshot of the printed results and inspect actual answers and arguments too.


In [ ]:
import subprocess, sys
RUN_LABEL = "baseline"
subprocess.run([sys.executable, "tool_choice_test.py", "--label", RUN_LABEL], check=True)


## 7. Download test evidence before ending the session
Download the JSON/CSV results for your report. Save your app screenshots separately.
The code submission goes on a public GitHub repository; this evidence ZIP is for keeping your own results.


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
results = Path("test_results")
evidence = sorted(results.glob("*.json")) + sorted(results.glob("*.csv"))
if not evidence:
    raise FileNotFoundError("No test evidence exists yet. Run the live test first.")
archive_path = Path("/content/ZaenDrive_test_results.zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for path in evidence:
        archive.write(path, arcname=path.name)
files.download(str(archive_path))


## Before handing in
Use the teacher's original Word template. Add the four real conversation screenshots, real routing-test score, public GitHub URL and your actual work time.
Export as **C115451101_Zaen_CustomerService.pdf**.
Upload app.py, agent.py, rental_tools.py, the CSV files, requirements.txt, .env.example, .gitignore, README.md and screenshots to GitHub. The tests may be included too.
Do **not** upload a real .env, database, key or credential-containing notebook output.
